# Solafune Satellite Precipitation Nowcasting — Kaggle Runner

Trains the Solafune nowcasting solution end-to-end on a Kaggle GPU (T4/P100/L4).

**Prerequisites**
1. Attach the `solafune-dataset` Kaggle Dataset — contains `solafune-train.zip` (18 GB) + `solafune-test.zip` (13 GB)
2. Enable **Internet** in notebook Settings (to `git clone` the repo)
3. Enable **GPU** (T4 / P100 / L4)

**Zero-extraction pipeline**
The two ZIPs stay compressed on the read-only `/kaggle/input` mount. The cache builder streams each TIF's bytes through `rasterio.MemoryFile` directly into the Zarr cache — nothing is ever extracted to disk. Peak `/kaggle/working` usage ≈ 10 GB (just the cache), safely under Kaggle's 20 GB quota.

**Run order**
1. Setup — clone repo, wire paths (no extraction)
2. Cache benchmark — Zarr vs memmap
3. Build train + eval caches by **streaming** from ZIPs
4. Compute normalization stats (also streamed)
5. Train (change `EXPERIMENT` to switch configs)
6. Inference + submission

**Session-safe** — the 12h wall? Re-run the training cell; `try_auto_resume()` picks up `last.pt`.

## 1. Setup

In [ ]:
# --- Clone repo from GitHub ---
REPO_URL = 'https://github.com/arnav-chauhan-kgpian/solafune.git'
REPO_DIR = '/kaggle/working/solafune'

# Kaggle Dataset mount (contains solafune-train.zip + solafune-test.zip)
DATASET_MOUNT = '/kaggle/input/solafune-dataset'
OUT_DIR = '/kaggle/working'

import os, sys, subprocess, zipfile
from pathlib import Path

# ---- 1. Clone (or pull) the code repo -----------------------------------
if not Path(REPO_DIR).exists():
    print('cloning repo...')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])
else:
    print('repo present; pulling latest...')
    try:
        subprocess.check_call(['git', '-C', REPO_DIR, 'pull', '--ff-only'])
    except subprocess.CalledProcessError:
        print('pull failed (probably no internet); continuing with cached copy')
sys.path.insert(0, REPO_DIR)

# ---- 2. Install missing pip packages ------------------------------------
for pkg in ('zarr', 'numcodecs', 'hydra-core', 'omegaconf'):
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# ---- 3. Resolve ZIP paths + extract each CSV to /kaggle/working ----------
# We don't extract the TIFs — but the CSVs are only inside the ZIPs, so we
# pull those two small files out into a writable location (a few MB each).
TRAIN_ZIP = Path(DATASET_MOUNT) / 'solafune-train.zip'
EVAL_ZIP  = Path(DATASET_MOUNT) / 'solafune-test.zip'
assert TRAIN_ZIP.exists(), f'missing {TRAIN_ZIP}'
assert EVAL_ZIP.exists(),  f'missing {EVAL_ZIP}'

CSV_DIR = Path(OUT_DIR) / 'csvs'
CSV_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = CSV_DIR / 'train_dataset.csv'
EVAL_CSV  = CSV_DIR / 'evaluation_target.csv'

def _extract_csv(zip_path: Path, csv_name: str, out_path: Path):
    if out_path.exists() and out_path.stat().st_size > 0:
        return
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # find entry whose basename == csv_name
        entry = next((n for n in zf.namelist() if n.endswith('/' + csv_name)
                                                 or n == csv_name), None)
        if entry is None:
            raise FileNotFoundError(f'{csv_name} not found in {zip_path}')
        with zf.open(entry) as src, open(out_path, 'wb') as dst:
            dst.write(src.read())

_extract_csv(TRAIN_ZIP, 'train_dataset.csv',    TRAIN_CSV)
_extract_csv(EVAL_ZIP,  'evaluation_target.csv', EVAL_CSV)
print('extracted CSVs:', TRAIN_CSV, EVAL_CSV)

CACHE_ROOT = Path(OUT_DIR) / 'cache'
NORM_STATS = CACHE_ROOT / 'norm_stats.json'
print('REPO_DIR: ', REPO_DIR)
print('TRAIN_ZIP:', TRAIN_ZIP, f'({TRAIN_ZIP.stat().st_size / 1e9:.1f} GB)')
print('EVAL_ZIP: ', EVAL_ZIP,  f'({EVAL_ZIP.stat().st_size / 1e9:.1f} GB)')
print('setup complete.')

## 2. Cache Backend Benchmark

In [ ]:
from src.data.cache.benchmark import run_benchmark
backend_json = CACHE_ROOT / 'backend.json'
if not backend_json.exists():
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    result = run_benchmark(output_path=backend_json, n_samples=200, channels=10, hw=81)
    print('recommended backend:', result.recommended)

import json
BACKEND = json.loads(backend_json.read_text())['recommended']
print('using backend:', BACKEND)

# On Kaggle we prefer Zarr for compression (disk-limited); switch here to override.
BACKEND = 'zarr'
print('locked backend to:', BACKEND)

## 3. Build Train + Eval Caches

In [ ]:
import pandas as pd
from src.data.cache import get_backend
from src.data.preprocessing import build_cache_spec
from src.data.preprocessing_zip import build_cache_from_zips

def _build_cache_from_zip_if_missing(csv, sat_zip, out_dir, load_gpm):
    spec_path = out_dir / 'spec.json'
    if spec_path.exists():
        print(f'cache exists: {out_dir}')
        return
    df = pd.read_csv(csv)
    spec, _ = build_cache_spec(df, out_dir, 'ir_only')
    cls = get_backend(BACKEND)
    b = cls(spec, compressor='lz4') if BACKEND == 'zarr' else cls(spec)
    build_cache_from_zips(
        csv_path=csv, sat_zip_path=sat_zip, gpm_zip_path=None,
        cache_root=out_dir, backend=b, band_mode='ir_only',
        load_gpm=load_gpm, verbose_every=2000,
    )
    b.close()
    print(f'cache built: {out_dir}')

# Train: streams TIFs (both satellite + GPM live inside solafune-train.zip)
_build_cache_from_zip_if_missing(TRAIN_CSV, TRAIN_ZIP, CACHE_ROOT / 'train', load_gpm=True)
# Eval: streams satellite TIFs only (GPM zeroed — Solafune eval placeholder)
_build_cache_from_zip_if_missing(EVAL_CSV,  EVAL_ZIP,  CACHE_ROOT / 'eval',  load_gpm=False)

## 4. Normalization Statistics

In [ ]:
from src.data.preprocessing_zip import compute_norm_stats_from_zip
from src.data.normalization import save_norm_stats

if not NORM_STATS.exists():
    NORM_STATS.parent.mkdir(parents=True, exist_ok=True)
    print('computing norm stats by streaming from train zip...')
    stats = compute_norm_stats_from_zip(
        zip_path=TRAIN_ZIP, csv_path=TRAIN_CSV,
        max_files_per_satellite=500, pixel_stride=2, seed=0,
    )
    save_norm_stats(NORM_STATS, stats)
print('norm stats:', NORM_STATS)

## 5. Training

Change `EXPERIMENT` and rerun this cell to iterate through the frozen experimental roadmap.

| EXPERIMENT | Hydra overrides applied |
|---|---|
| `exp0_baseline` | strong baseline (log1p + rain-weighted + BCE + stratified sampling + EMA) |
| `exp1_rain_weighted` | `loss.rain_weighted=true loss.rain_weight_scale=3.0` |
| `exp2_conv3d` | `+model/temporal=conv3d` |
| `exp3_nll` | `+model/heads=probabilistic loss.nll_weight=0.3` |
| `exp4_sampling` | `data.sampling.strategy=combined` (loc × precip) |
| `exp5_band_dropout` | `augmentation=full` with `band_dropout_prob=0.3` |
| `exp6_efficientnet` | `+model/encoder=efficientnet_b3` |
| `exp7_seed_ensemble` | second seed (run with `SEED=43`) — then average at inference |

In [ ]:
EXPERIMENT = 'exp0_baseline'    # <── change here to sweep experiments
EPOCHS     = 50
SEED       = 42

TRAINING_DIR = Path(OUT_DIR) / 'training' / EXPERIMENT / f'seed_{SEED}'
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

# Compose config via Hydra
from hydra import compose, initialize_config_dir
import hydra
hydra_dir = str(Path(REPO_DIR) / 'configs')
if hydra.core.global_hydra.GlobalHydra.instance().is_initialized():
    hydra.core.global_hydra.GlobalHydra.instance().clear()

# Only cache_dir + csv paths + norm_stats matter for training; the raw TIF
# roots were used at cache-build time (streamed from ZIP) and are no longer
# needed. We still set them to keep the Hydra schema happy.
overrides = [
    f'data.cache_dir={CACHE_ROOT}/train',
    f'data.norm_stats_path={NORM_STATS}',
    f'data.train_csv={TRAIN_CSV}',
    f'data.eval_csv={EVAL_CSV}',
    f'data.train_root={CACHE_ROOT}/train',
    f'data.eval_root={CACHE_ROOT}/eval',
    f'data.cache.backend={BACKEND}',
    f'training.output_dir={TRAINING_DIR}',
    f'training.epochs={EPOCHS}',
    f'seed={SEED}',
    f'+experiment={EXPERIMENT}',
]
with initialize_config_dir(config_dir=hydra_dir, version_base=None):
    cfg = compose(config_name='config', overrides=overrides)

from omegaconf import OmegaConf
print(OmegaConf.to_yaml(cfg))

In [ ]:
# Build datasets, model, trainer
from src.constants import max_active_channels
from src.data.dataloader import DataLoaderConfig, build_dataloader, build_sampler
from src.data.dataset import DatasetConfig, SolafuneDataset, split_indices_by_location
from src.experiment.tracker import snapshot_run
from src.models import build_model
from src.seed import seed_everything
from src.training.losses import build_loss
from src.training.schedulers import build_optimizer, build_scheduler
from src.training.trainer import Trainer, TrainerConfig

seed_everything(int(cfg.seed))
snapshot_run(TRAINING_DIR, cfg, repo_root=Path(REPO_DIR))

ds_cfg = DatasetConfig(
    cache_dir=Path(cfg.data.cache_dir),
    csv_path=Path(cfg.data.train_csv),
    norm_stats_path=Path(cfg.data.norm_stats_path),
    image_size=int(cfg.data.image_size),
    bands=str(cfg.data.bands),
    include_diff_frames=bool(cfg.data.include_diff_frames),
    missing_frame_strategy=str(cfg.data.missing_frame_strategy),
    cache_backend=BACKEND,
)
df = pd.read_csv(ds_cfg.csv_path)
train_idx, val_idx = split_indices_by_location(df, list(cfg.data.val_locations))
train_ds = SolafuneDataset(ds_cfg, df=df, indices=train_idx)
val_ds   = SolafuneDataset(ds_cfg, df=df, indices=val_idx)
print(f'train: {len(train_ds)}   val: {len(val_ds)}')

dl = DataLoaderConfig(
    batch_size=int(cfg.data.batch_size), num_workers=int(cfg.data.num_workers),
    pin_memory=True, persistent_workers=True, prefetch_factor=2, drop_last=True,
)
train_loader = build_dataloader(
    train_ds, dl,
    sampler=build_sampler(train_ds, str(cfg.data.sampling.strategy),
                          precip_weight_scale=float(cfg.data.sampling.precip_weight_scale)),
    base_seed=int(cfg.seed),
)
val_loader = build_dataloader(val_ds, dl, shuffle=False, base_seed=int(cfg.seed))

mcfg = OmegaConf.to_container(cfg.model, resolve=True)
mcfg['in_channels_per_frame'] = max_active_channels(str(cfg.data.bands))
mcfg['n_frames'] = int(cfg.data.frames)
mcfg['n_diff_frames'] = int(cfg.data.frames - 1) if cfg.data.include_diff_frames else 0
model = build_model(mcfg)
print(f'params: {sum(p.numel() for p in model.parameters()):,}')

loss_fn = build_loss(OmegaConf.to_container(cfg.loss))
opt = build_optimizer(model, OmegaConf.to_container(cfg.optimizer))
sched, seb = build_scheduler(
    opt, OmegaConf.to_container(cfg.scheduler),
    steps_per_epoch=len(train_loader), epochs=int(cfg.training.epochs),
)
tcfg = TrainerConfig(**OmegaConf.to_container(cfg.training))
tcfg.step_scheduler_each_batch = seb
trainer = Trainer(model, opt, sched, loss_fn, train_loader, val_loader, tcfg)
trainer.try_auto_resume()   # session-safe: pick up last.pt if present

In [ ]:
# Kick off training. Progress is streamed via the logger + saved to CSV/TensorBoard.
best_val = trainer.fit()
print('best val metric:', best_val)

## 6. Diagnostic plots

In [ ]:
from src.visualization import plot_training_curves, plot_val_curves
plot_training_curves(TRAINING_DIR / 'train_metrics.csv', TRAINING_DIR / 'plots' / 'train.png')
plot_val_curves(TRAINING_DIR / 'val_metrics.csv', TRAINING_DIR / 'plots' / 'val.png')
print('plots saved in', TRAINING_DIR / 'plots')

## 7. Inference + Submission

In [ ]:
from src.inference.predict import PredictionConfig, predict
from src.inference.submission import write_submission
from src.training.callbacks import CheckpointSaver

# Load best (EMA-preferred) checkpoint
best_ckpt = TRAINING_DIR / 'checkpoints' / 'best.pt'
assert best_ckpt.exists(), best_ckpt
state = torch.load(best_ckpt, map_location='cpu', weights_only=False)
if state.get('ema') is not None:
    # Load EMA weights (shadow → model)
    from src.training.ema import ExponentialMovingAverage
    ema = ExponentialMovingAverage(model, decay=cfg.training.ema_decay)
    ema.load_state_dict(state['ema'])
    ema.apply(model).__enter__()   # leave model in EMA weights for inference
    print('loaded EMA weights')
else:
    model.load_state_dict(state['model'])
    print('loaded raw weights')

# Build eval dataset
eval_ds_cfg = DatasetConfig(
    cache_dir=CACHE_ROOT / 'eval',
    csv_path=EVAL_CSV,
    norm_stats_path=NORM_STATS,
    image_size=int(cfg.data.image_size), bands='ir_only',
    include_diff_frames=True, cache_backend=BACKEND,
)
eval_ds = SolafuneDataset(eval_ds_cfg)
eval_loader = build_dataloader(
    eval_ds,
    DataLoaderConfig(batch_size=32, num_workers=2, pin_memory=True,
                     persistent_workers=True, prefetch_factor=2, drop_last=False),
    shuffle=False, base_seed=int(cfg.seed),
)
print(f'eval samples: {len(eval_ds)}')

preds = predict(
    model, eval_loader,
    PredictionConfig(amp=True, tta=True, rain_mask_threshold=0.15),
)
print(f'predictions shape: {preds.shape}   finite: {bool((preds == preds).all())}   min/max: {preds.min():.3f}/{preds.max():.3f}')

In [ ]:
# Write submission GeoTIFFs (identity CRS, float32, 41×41, LZW-compressed)
SUB_DIR = Path(OUT_DIR) / 'submission'
TEST_FILES = SUB_DIR / 'test_files'
n_written = write_submission(preds, EVAL_CSV, TEST_FILES)

# Copy evaluation_target.csv into the submission directory per Solafune format
import shutil
shutil.copy(EVAL_CSV, SUB_DIR / 'evaluation_target.csv')

# Zip archive for upload
archive = shutil.make_archive(str(Path(OUT_DIR) / 'submission'), 'zip', str(SUB_DIR))
print(f'files written: {n_written}')
print(f'archive: {archive}   size: {Path(archive).stat().st_size / 1e6:.1f} MB')

In [ ]:
# Final format audit — reads a random subset back and asserts the Solafune contract
import random, numpy as np
from src.utils.io import read_gpm_tif
random.seed(0)
sample_names = random.sample(list(pd.read_csv(EVAL_CSV)['gpm_imerg_filename']), 25)
for name in sample_names:
    arr, meta = read_gpm_tif(TEST_FILES / name)
    assert arr.shape == (41, 41), (name, arr.shape)
    assert arr.dtype == np.float32, (name, arr.dtype)
    assert np.isfinite(arr).all(), name
print(f'audit passed: 25/25 sampled TIFs conform to (41,41) float32 finite')
print('submission ready at:', SUB_DIR)

## 8. Experiment sweep helper (optional)

Runs the frozen roadmap sequentially inside a single Kaggle session. Each experiment
starts from the previous best checkpoint via `try_auto_resume()`.

**Warning**: A full sweep is ~40+ GPU hours. Only enable if you have a fresh session and expect the compute budget.

```python
SWEEP = ['exp0_baseline', 'exp2_conv3d', 'exp3_nll']
for exp in SWEEP:
    print(f'\n=== running {exp} ===')
    # 1. rerun cell 5 with EXPERIMENT=exp
    # 2. rerun cell 7 to produce a per-experiment submission
    # (in a notebook: just set EXPERIMENT and click Run All from cell 5 downward)
```